### 📦 IMPORTACIONES NECESARIAS

In [1]:
import pandas as pd
import requests
import glob
import os
from time import sleep


### 🧠 CONFIGURACIÓN GENERAL
### ------------------------------------------------

In [ ]:
PROMPT_TEMPLATE = "<s> [INST] Clasificá el siguiente abstract: {text} [/INST]"
BATCH_SIZE = 10  # 🔁 Cantidad de abstracts por lote
INPUT_FOLDER = "csv_inputs"  # 📁 Carpeta donde están los CSVs de entrada
OUTPUT_FOLDER = "csv_outputs"  # 📁 Carpeta donde se guardarán los resultados
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

### 🧠💬 FUNCIÓN DE LLAMADO A LA API DE OLLAMA MIXTRAL

In [ ]:
def classify_abstract(text: str, model="mixtral") -> str:
    try:
        prompt = PROMPT_TEMPLATE.format(text=text.strip())
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=120
        )
        return response.json().get("response", "[Error: sin respuesta]")
    except Exception as e:
        return f"[Error: {str(e)}]"


### 📄 FUNCIÓN PARA CARGAR CSVs

In [ ]:
def load_all_csvs(folder):
    csv_files = glob.glob(os.path.join(folder, "*.csv"))
    all_data = pd.DataFrame()
    for file in csv_files:
        df = pd.read_csv(file)
        df.columns = df.columns.str.strip()
        df = df.rename(columns={df.columns[0]: "PMID", df.columns[1]: "Abstract"})
        all_data = pd.concat([all_data, df], ignore_index=True)
    return all_data

### 💾 FUNCIÓN PARA GUARDAR ARCHIVO AUTOMÁTICAMENTE

In [ ]:
def save_batch(df_partial, batch_number):
    filename = os.path.join(OUTPUT_FOLDER, f"ai_classified_DB_{batch_number:03d}.csv")
    df_partial.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"[✔] Guardado automático: {filename}")

### 💾 SECCIÓN PARA GUARDAR MANUALMENTE SI SE DESEA

In [ ]:
def manual_save(df_final, name="ai_classified_DB_manual.csv"):
    filename = os.path.join(OUTPUT_FOLDER, name)
    df_final.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"[💾] Guardado manual en: {filename}")

### 🚀 EJECUCIÓN PRINCIPAL DEL PROCESO

In [ ]:
def main():
    df = load_all_csvs(INPUT_FOLDER)
    results = []
    batch_number = 1

    for i in range(0, len(df), BATCH_SIZE):
        batch = df.iloc[i:i+BATCH_SIZE]
        classified = []

        print(f"[→] Procesando lote {batch_number} ({i}-{i+BATCH_SIZE})...")

        for _, row in batch.iterrows():
            classification = classify_abstract(row["Abstract"])
            classified.append({
                "PMID": row["PMID"],
                "Abstract": row["Abstract"],
                "Clasificación": classification
            })

        df_batch = pd.DataFrame(classified)
        results.append(df_batch)
        save_batch(df_batch, batch_number)
        batch_number += 1

        # Pequeña pausa entre lotes opcional
        sleep(1)

    # Guardado total final (por si querés un archivo unificado también)
    df_total = pd.concat(results, ignore_index=True)
    manual_save(df_total)


### 🔄 EJECUTAR SI EL SCRIPT ES PRINCIPAL

In [ ]:
if __name__ == "__main__":
    main()